In [1]:
# ============================================================
# ABLATION STUDY: 8 Experiments
# Fixed:    loss_cls + loss_dc + loss_dh
# Ablated:  loss_orth | loss_ent | loss_pseudo
#
# Exp | orth | ent | pseudo
#  1  |  Y   |  Y  |   Y     ← full model (baseline)
#  2  |  Y   |  Y  |   N
#  3  |  Y   |  N  |   Y
#  4  |  N   |  Y  |   Y
#  5  |  Y   |  N  |   N
#  6  |  N   |  Y  |   N
#  7  |  N   |  N  |   Y
#  8  |  N   |  N  |   N     ← minimal model
# ============================================================


# ============================================================
# SECTION 0: Imports & Config
# ============================================================

import os
import sys
import copy
import math
import random
import itertools
from collections import deque

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

sys.path.append(r"D:\Woodchip_moisture_content\Dataset")
from dataset_1_2 import SourceDataset, TargetDataset, WoodChipTargetEvalDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

BASE_SAVE_DIR = r"D:\Woodchip_moisture_content\My_model\Sir\ablation"
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP  = torch.cuda.is_available()

BACKBONE_NAME = "convnext_small"
NUM_CLASSES   = 3
FEATURE_DIM   = 512
BATCH_SIZE    = 16
NUM_EPOCHS    = 60
NUM_WORKERS   = 0
DROPOUT       = 0.2

BACKBONE_LR  = 1e-5
HEAD_LR      = 1e-4
WEIGHT_DECAY = 1e-4

LAMBDA_COMMON     = 0.5
LAMBDA_HETERO     = 0.1
LAMBDA_ORTH       = 0.01
LAMBDA_COMMON_MIN = 0.05
LAMBDA_COMMON_MAX = 1.00

FREEZE_BACKBONE_EPOCHS = 2
DBS_MA_WINDOW          = 3
UDA_ALPHA              = 10.0
FOCAL_GAMMA            = 2.0

LAMBDA_ENT         = 0.1
PSEUDO_START_EPOCH = 10
PSEUDO_THRESHOLD   = 0.85
LAMBDA_PSEUDO      = 0.3

LABEL_NAMES = ['Dry', 'Medium', 'Wet']

# ---- All 8 ablation configs ----
# (use_orth, use_ent, use_pseudo)
ABLATION_CONFIGS = list(itertools.product([True, False], repeat=3))
# itertools.product gives all 8 combinations in order:
# (T,T,T), (T,T,F), (T,F,T), (F,T,T), (T,F,F), (F,T,F), (F,F,T), (F,F,F)

def config_name(use_orth, use_ent, use_pseudo):
    parts = []
    parts.append("orth" if use_orth   else "noOrth")
    parts.append("ent"  if use_ent    else "noEnt")
    parts.append("psl"  if use_pseudo else "noPsl")
    return "_".join(parts)


# ============================================================
# SECTION 1: Dataset Setup
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class TransformWrapper(Dataset):
    def __init__(self, base_dataset, transform):
        self.samples   = base_dataset.samples
        self.transform = transform
        self.labeled   = isinstance(self.samples[0], tuple)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.labeled:
            path, label = self.samples[idx]
            img = Image.open(path).convert('RGB')
            return self.transform(img), label
        else:
            path = self.samples[idx]
            img = Image.open(path).convert('RGB')
            return self.transform(img)


src_train_ds = TransformWrapper(SourceDataset(),             train_transform)
src_eval_ds  = TransformWrapper(SourceDataset(),             eval_transform)
tgt_train_ds = TransformWrapper(TargetDataset(),             train_transform)
tgt_eval_ds  = TransformWrapper(WoodChipTargetEvalDataset(), eval_transform)

src_loader      = DataLoader(src_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
tgt_loader      = DataLoader(tgt_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
src_eval_loader = DataLoader(src_eval_ds,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
tgt_eval_loader = DataLoader(tgt_eval_ds,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

print(f"Source train : {len(src_train_ds)} | Target train : {len(tgt_train_ds)} | Target eval : {len(tgt_eval_ds)}")


# ============================================================
# SECTION 2: Model Architecture
# ============================================================

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)


class AdvancedBackbone(nn.Module):
    def __init__(self, backbone_name="convnext_small", pretrained=True):
        super().__init__()
        self.backbone_name = backbone_name.lower()

        if self.backbone_name == "convnext_small":
            weights = models.ConvNeXt_Small_Weights.IMAGENET1K_V1 if pretrained else None
            b = models.convnext_small(weights=weights)
            self.features    = b.features
            self.pool        = nn.AdaptiveAvgPool2d((1, 1))
            self.feature_dim = b.classifier[2].in_features

        elif self.backbone_name == "resnet50":
            weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            b = models.resnet50(weights=weights)
            self.features    = nn.Sequential(*list(b.children())[:-1])
            self.feature_dim = b.fc.in_features

        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")

    def forward(self, x):
        if self.backbone_name == "convnext_small":
            x = self.features(x)
            x = self.pool(x)
            return torch.flatten(x, 1)
        elif self.backbone_name == "resnet50":
            return torch.flatten(self.features(x), 1)


class WoodChipUDA(nn.Module):
    def __init__(self, backbone_name="convnext_small", feature_dim=512,
                 num_classes=3, dropout=0.2, pretrained=True):
        super().__init__()
        self.backbone = AdvancedBackbone(backbone_name, pretrained)
        in_dim = self.backbone.feature_dim

        self.common_encoder = nn.Sequential(
            nn.Linear(in_dim, feature_dim), nn.LayerNorm(feature_dim),
            nn.GELU(), nn.Dropout(dropout)
        )
        self.hetero_encoder = nn.Sequential(
            nn.Linear(in_dim, feature_dim), nn.LayerNorm(feature_dim),
            nn.GELU(), nn.Dropout(dropout)
        )
        self.classifier    = nn.Linear(feature_dim, num_classes)
        self.domain_common = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(256, 2)
        )
        self.domain_hetero = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(256, 2)
        )

    def forward(self, x, grl_lambda=1.0):
        f  = self.backbone(x)
        zc = self.common_encoder(f)
        zh = self.hetero_encoder(f)
        y_logits        = self.classifier(zc)
        d_common_logits = self.domain_common(grad_reverse(zc, grl_lambda))
        d_hetero_logits = self.domain_hetero(zh)
        return y_logits, d_common_logits, d_hetero_logits, zc, zh


# ============================================================
# SECTION 3: Loss Functions
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt      = torch.exp(-ce_loss)
        return ((1.0 - pt) ** self.gamma * ce_loss).mean()

focal_loss = FocalLoss(gamma=FOCAL_GAMMA)

def entropy_loss(logits):
    probs     = F.softmax(logits, dim=1)
    log_probs = F.log_softmax(logits, dim=1)
    return -torch.sum(probs * log_probs, dim=1).mean()

def orthogonality_loss(zc, zh):
    zc = F.normalize(zc, dim=1)
    zh = F.normalize(zh, dim=1)
    return torch.mean((zc * zh) ** 2)

def compute_dbs(common_acc, hetero_acc):
    return hetero_acc - abs(common_acc - 50.0)

def compute_uda_score(dbs_ma, cls_loss, alpha=10.0):
    return dbs_ma - alpha * cls_loss


# ============================================================
# SECTION 4: Helpers
# ============================================================

@torch.no_grad()
def evaluate_target(model, loader, device, use_amp):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", enabled=use_amp):
            y_logits, _, _, _, _ = model(x, grl_lambda=0.0)
        correct += (y_logits.argmax(1) == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total


@torch.no_grad()
def evaluate_domain_separation(model, src_loader, tgt_loader, device, use_amp):
    model.eval()
    common_correct = hetero_correct = total = 0
    src_iter = iter(src_loader)
    tgt_iter = iter(tgt_loader)

    for _ in range(min(len(src_loader), len(tgt_loader))):
        xs, _ = next(src_iter)
        xt     = next(tgt_iter)
        if isinstance(xt, (list, tuple)):
            xt = xt[0]

        x = torch.cat([xs.to(device), xt.to(device)], dim=0)
        domain_labels = torch.cat([
            torch.zeros(xs.size(0), dtype=torch.long),
            torch.ones(xt.size(0),  dtype=torch.long)
        ]).to(device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            _, d_common, d_hetero, _, _ = model(x, grl_lambda=0.0)

        common_correct += (d_common.argmax(1) == domain_labels).sum().item()
        hetero_correct += (d_hetero.argmax(1) == domain_labels).sum().item()
        total          += domain_labels.size(0)

    return 100.0 * common_correct / total, 100.0 * hetero_correct / total


def compute_metrics(model, loader, device, use_amp):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits, _, _, _, _ = model(x, grl_lambda=0.0)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.numpy())
    y_true, y_pred = np.array(all_labels), np.array(all_preds)
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "recall":    recall_score(y_true, y_pred, average='weighted', zero_division=0),
        "f1":        f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

def set_backbone_trainable(model, trainable):
    for p in model.backbone.parameters():
        p.requires_grad = trainable


# ============================================================
# SECTION 5: Single Experiment Runner
# ============================================================

def run_experiment(exp_id, use_orth, use_ent, use_pseudo):
    name     = config_name(use_orth, use_ent, use_pseudo)
    save_dir = os.path.join(BASE_SAVE_DIR, f"exp{exp_id:02d}_{name}")
    os.makedirs(save_dir, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"EXP {exp_id}/8  |  {name}")
    print(f"  orth={use_orth} | ent={use_ent} | pseudo={use_pseudo}")
    print(f"{'='*70}")

    # Fresh model + optimizer each experiment
    torch.manual_seed(SEED)
    model = WoodChipUDA(
        backbone_name=BACKBONE_NAME,
        feature_dim=FEATURE_DIM,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
        pretrained=True
    ).to(DEVICE)

    optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(),       "lr": BACKBONE_LR},
        {"params": model.common_encoder.parameters(), "lr": HEAD_LR},
        {"params": model.hetero_encoder.parameters(), "lr": HEAD_LR},
        {"params": model.classifier.parameters(),     "lr": HEAD_LR},
        {"params": model.domain_common.parameters(),  "lr": HEAD_LR},
        {"params": model.domain_hetero.parameters(),  "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    dbs_queue      = deque(maxlen=DBS_MA_WINDOW)
    best_uda_score = -1e9
    lambda_common  = LAMBDA_COMMON
    history        = []

    for epoch in range(1, NUM_EPOCHS + 1):

        set_backbone_trainable(model, epoch > FREEZE_BACKBONE_EPOCHS)
        model.train()

        src_iter = iter(src_loader)
        tgt_iter = iter(tgt_loader)
        num_iter = min(len(src_loader), len(tgt_loader))

        ep_cls = ep_dc = ep_dh = ep_orth = ep_ent = ep_pseudo = 0.0

        p          = epoch / NUM_EPOCHS
        grl_lambda = 0.5 * (2.0 / (1.0 + math.exp(-10 * p)) - 1.0)

        for _ in range(num_iter):
            xs, ys = next(src_iter)
            xt     = next(tgt_iter)
            if isinstance(xt, (list, tuple)):
                xt = xt[0]

            xs = xs.to(DEVICE, non_blocking=True)
            ys = ys.to(DEVICE, non_blocking=True)
            xt = xt.to(DEVICE, non_blocking=True)

            x = torch.cat([xs, xt], dim=0)
            domain_labels = torch.cat([
                torch.zeros(xs.size(0), dtype=torch.long),
                torch.ones(xt.size(0),  dtype=torch.long)
            ]).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                y_logits, d_common, d_hetero, zc, zh = model(x, grl_lambda)

                src_logits = y_logits[:xs.size(0)]
                tgt_logits = y_logits[xs.size(0):]

                # --- Fixed losses (always on) ---
                loss_cls = focal_loss(src_logits, ys)
                loss_dc  = F.cross_entropy(d_common, domain_labels)
                loss_dh  = F.cross_entropy(d_hetero, domain_labels)

                # --- Ablated losses (conditional) ---
                loss_orth   = orthogonality_loss(zc, zh) if use_orth   else torch.tensor(0.0, device=DEVICE)
                loss_ent    = entropy_loss(tgt_logits)   if use_ent    else torch.tensor(0.0, device=DEVICE)
                loss_pseudo = torch.tensor(0.0, device=DEVICE)

                if use_pseudo and epoch > PSEUDO_START_EPOCH:
                    tgt_probs_det = F.softmax(tgt_logits.detach(), dim=1)
                    max_probs, pseudo_labels = tgt_probs_det.max(dim=1)
                    confident_mask = max_probs >= PSEUDO_THRESHOLD
                    if confident_mask.sum() >= 2:
                        loss_pseudo = focal_loss(
                            tgt_logits[confident_mask],
                            pseudo_labels[confident_mask]
                        )

                loss = (loss_cls
                        + lambda_common * loss_dc
                        + LAMBDA_HETERO  * loss_dh
                        + LAMBDA_ORTH    * loss_orth
                        + LAMBDA_ENT     * loss_ent
                        + LAMBDA_PSEUDO  * loss_pseudo)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

            ep_cls    += loss_cls.item()
            ep_dc     += loss_dc.item()
            ep_dh     += loss_dh.item()
            ep_orth   += loss_orth.item()
            ep_ent    += loss_ent.item()
            ep_pseudo += loss_pseudo.item()

        scheduler.step()

        avg_cls = ep_cls / num_iter

        target_acc              = evaluate_target(model, tgt_eval_loader, DEVICE, USE_AMP)
        common_acc, hetero_acc  = evaluate_domain_separation(model, src_loader, tgt_loader, DEVICE, USE_AMP)
        raw_dbs                 = compute_dbs(common_acc, hetero_acc)
        dbs_queue.append(raw_dbs)
        smooth_dbs              = sum(dbs_queue) / len(dbs_queue)
        uda_score               = compute_uda_score(smooth_dbs, avg_cls, UDA_ALPHA)

        if common_acc > 60.0:
            lambda_common = min(lambda_common * 1.05, LAMBDA_COMMON_MAX)
        elif common_acc < 40.0:
            lambda_common = max(lambda_common * 0.95, LAMBDA_COMMON_MIN)

        history.append({
            "epoch": epoch, "target_acc": target_acc,
            "cls_loss": avg_cls, "raw_dbs": raw_dbs,
            "smooth_dbs": smooth_dbs, "uda_score": uda_score,
        })

        if uda_score > best_uda_score:
            best_uda_score = uda_score
            torch.save(
                copy.deepcopy(model.state_dict()),
                os.path.join(save_dir, "best_model.pth")
            )

        print(
            f"  Ep {epoch:02d} | TargetAcc: {target_acc:.2f}% | "
            f"Cls: {avg_cls:.4f} | DBS: {raw_dbs:.2f} | UDA: {uda_score:.2f}"
        )

    pd.DataFrame(history).to_csv(os.path.join(save_dir, "history.csv"), index=False)

    # Load best model → compute final metrics
    model.load_state_dict(torch.load(os.path.join(save_dir, "best_model.pth"), map_location=DEVICE))
    tgt_metrics = compute_metrics(model, tgt_eval_loader, DEVICE, USE_AMP)
    src_metrics = compute_metrics(model, src_eval_loader, DEVICE, USE_AMP)

    result = {
        "exp_id":      exp_id,
        "name":        name,
        "use_orth":    use_orth,
        "use_ent":     use_ent,
        "use_pseudo":  use_pseudo,
        "tgt_accuracy":  tgt_metrics["accuracy"],
        "tgt_precision": tgt_metrics["precision"],
        "tgt_recall":    tgt_metrics["recall"],
        "tgt_f1":        tgt_metrics["f1"],
        "src_accuracy":  src_metrics["accuracy"],
        "best_uda_score": best_uda_score,
    }

    print(f"\n  RESULT: Target Acc={tgt_metrics['accuracy']:.3f} | F1={tgt_metrics['f1']:.3f}")
    return result


# ============================================================
# SECTION 6: Run All 8 Experiments
# ============================================================

all_results = []

for exp_id, (use_orth, use_ent, use_pseudo) in enumerate(ABLATION_CONFIGS, start=1):
    result = run_experiment(exp_id, use_orth, use_ent, use_pseudo)
    all_results.append(result)

# ============================================================
# SECTION 7: Summary Table
# ============================================================

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("tgt_accuracy", ascending=False).reset_index(drop=True)

summary_path = os.path.join(BASE_SAVE_DIR, "ablation_summary.csv")
results_df.to_csv(summary_path, index=False)

print("\n" + "=" * 80)
print("ABLATION STUDY — FINAL SUMMARY (sorted by Target Accuracy)")
print("=" * 80)
print(f"{'Exp':<5} {'Config':<25} {'Tgt Acc':>9} {'Tgt F1':>8} {'Src Acc':>9} {'UDA Score':>11}")
print("-" * 80)
for _, row in results_df.iterrows():
    print(
        f"{int(row.exp_id):<5} {row['name']:<25} "
        f"{row.tgt_accuracy:>9.3f} {row.tgt_f1:>8.3f} "
        f"{row.src_accuracy:>9.3f} {row.best_uda_score:>11.2f}"
    )
print("=" * 80)
print(f"\nBest config : {results_df.iloc[0]['name']}")
print(f"Best Tgt Acc: {results_df.iloc[0]['tgt_accuracy']:.3f}")
print(f"\nFull results saved to: {summary_path}")

Source train : 800 | Target train : 800 | Target eval : 800

EXP 1/8  |  orth_ent_psl
  orth=True | ent=True | pseudo=True
  Ep 01 | TargetAcc: 34.00% | Cls: 0.3356 | DBS: 57.75 | UDA: 54.39
  Ep 02 | TargetAcc: 47.62% | Cls: 0.1795 | DBS: 56.62 | UDA: 55.39
  Ep 03 | TargetAcc: 58.38% | Cls: 0.1343 | DBS: 57.94 | UDA: 56.09
  Ep 04 | TargetAcc: 45.88% | Cls: 0.1023 | DBS: 61.88 | UDA: 57.79
  Ep 05 | TargetAcc: 47.00% | Cls: 0.0911 | DBS: 94.75 | UDA: 70.61
  Ep 06 | TargetAcc: 45.88% | Cls: 0.0825 | DBS: 85.94 | UDA: 80.03
  Ep 07 | TargetAcc: 62.50% | Cls: 0.0671 | DBS: 94.62 | UDA: 91.10
  Ep 08 | TargetAcc: 81.12% | Cls: 0.0777 | DBS: 87.31 | UDA: 88.51
  Ep 09 | TargetAcc: 73.62% | Cls: 0.0693 | DBS: 94.50 | UDA: 91.45
  Ep 10 | TargetAcc: 80.75% | Cls: 0.0521 | DBS: 66.19 | UDA: 82.15
  Ep 11 | TargetAcc: 82.00% | Cls: 0.0678 | DBS: 96.56 | UDA: 85.07
  Ep 12 | TargetAcc: 74.62% | Cls: 0.0623 | DBS: 92.44 | UDA: 84.44
  Ep 13 | TargetAcc: 60.38% | Cls: 0.0690 | DBS: 99.38 | UDA: